# 82514 · Sesión S24 — Copiar al experto: la deriva, la media de dos modos y la orden

**Bloque 8** · lunes 14 de diciembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** El robot del almacén tiene que rodear un palé. Un operario lo conduce 40 veces con el mando y una red de 64 × 64 copia sus pares (posición, acción): la clonación de comportamiento. Un empujón la lleva donde nadie estuvo y choca; DAgger lo arregla. Si unos operarios rodean por arriba y otros por abajo, la red da la media de las dos, como el 2R de S10 y los rumbos en torno a 180° de S19; la difusión y una orden lo arreglan.

**Se apoya en:** Ross, Gordon y Bagnell (2011), DAgger; Chi et al. (2023), la política de difusión; S10 (el 2R del puesto) y S19 (la media circular); documentación de scikit-learn.

**Cómo usarlo en clase.** Sigue el guion de la sesión S24 en las diapositivas del bloque 8. Ejecuta la primera celda una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos segundos.
import os, sys, importlib, importlib.metadata as md, subprocess
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')    # pygame (el dibujo del péndulo) sin pantalla y sin avisos

FIJAS = {'gymnasium': '1.4.0', 'scikit-learn': '1.6.1', 'stable-baselines3': '2.9.0'}   # las del bloque 8
COLAB = {'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0', 'torch': '2.11.0'}  # se comprueban, no se instalan
USA = ['scikit-learn']             # las del bloque que usa este cuaderno
EN_COLAB = 'google.colab' in sys.modules

def version(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

for paquete in USA:
    if paquete == 'stable-baselines3' and version('torch') is None:
        print('AVISO: no hay torch en este entorno: la sección de PPO no se ejecuta (Colab sí lo trae).')
        continue
    if version(paquete) != FIJAS[paquete]:
        print('Instalando', paquete, FIJAS[paquete], '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{paquete}=={FIJAS[paquete]}'], check=False)
importlib.invalidate_caches()

import math, random, time
import numpy as np, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
for p in ['numpy', 'scipy', 'matplotlib'] + USA:
    w = (version(p) or 'no instalada').split('+')[0]
    v = {**COLAB, **FIJAS}[p]
    print(f'{p:18s} {w:12s}', '' if w == v else f'(probado con {v})')
print('en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

plt.rcParams['figure.figsize'] = (11, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def pc(x):
    """Una fracción como porcentaje entero."""
    return f'{100 * x:.0f} %'


import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings('ignore', category=ConvergenceWarning)   # la difusión para a las 60 vueltas a propósito

> **Aviso para Colab.** Ábrelo al empezar la sesión y ejecuta la primera celda: comprueba scikit-learn 1.6.1 (Colab ya la trae) y tarda unos segundos.
>
> **En clase (tres minutos):** la sección 1 sin detenerte, y el taller: la 2 y la 4. Las cifras tienen que salir como en las diapositivas. Si al volver a él Colab dice que se ha desconectado (pasa tras un rato largo sin usarlo), conéctalo de nuevo (arriba a la derecha) y ejecuta otra vez la primera celda y la de la sección 1: unos segundos.
>
> **En casa:** la sección 3 (DAgger), la 5 (la difusión, unos 10 s), la 6 (la orden) y los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle.

## 1. Un operario rodea un palé 40 veces, y una red lo copia (diapositiva 25)

*En clase, ejecuta esta celda sin detenerte (unos segundos).*

El robot del almacén (0,28 m de radio, 0,5 m/s) toma una decisión cada 0,1 s: pasos de 5 cm. Sale de cerca de (0; 0) y va a la meta, (4; 0); en medio hay un palé de 0,8 × 1,2 m centrado en (2; 0). Como en S22, el palé se infla con el radio del robot: choca si su centro entra en la zona rosa.

Un operario lo conduce con el mando (teleoperación, como el guiado manual de S4): va a la esquina de antes del palé, pasa por arriba a 1,03 m del eje y baja a la meta. El mando tiene un poco de ruido. De cada paso se guarda un par (posición, acción).

La **clonación de comportamiento** es aprendizaje supervisado con esos pares: una red de 64 × 64 (la de S17, más grande) de la posición a la acción. Sin recompensa y sin explorar.

El código es el mismo que dio las cifras de las diapositivas.

In [ ]:
# El palé, el operario, la copia, DAgger, la difusión y las medias (el mismo código que las diapositivas)
# El robot del almacén (0,28 m de radio) rodea un palé en un pasillo. Una decisión cada 0,1 s a 0,5 m/s: pasos de 5 cm.
# El palé, de 0,8 × 1,2 m, está atravesado con el centro en (2,0; 0,0); el robot sale de cerca de (0; 0) y va a la
# meta, (4,0; 0,0). Choca si su centro se acerca a menos de 0,28 m del palé (el palé inflado con el radio, como en S22).
PASO, R_PALE = 0.05, 0.28          # el paso (m) y el radio del robot del almacén (m)
PALE = (2.0, 0.0, 0.4, 0.6)        # centro x, centro y, media anchura (x) y media longitud (y) del palé, en m
META = np.array([4.0, 0.0])
HOLGURA = 0.15                     # lo que deja entre el palé inflado y su camino
Y_PASO = PALE[3] + R_PALE + HOLGURA          # 1,03 m: la altura a la que pasa junto al palé
X_ANTES = PALE[0] - PALE[2] - HOLGURA        # 1,45 m: la esquina de antes del palé
X_DESPUES = PALE[0] + PALE[2] + HOLGURA      # 2,55 m: la de después


def unitario(v):
    return v / (np.linalg.norm(v) + 1e-12)


def al_pale(p):
    """La distancia (m) del centro del robot al palé."""
    dx = max(abs(p[0] - PALE[0]) - PALE[2], 0.0)
    dy = max(abs(p[1] - PALE[1]) - PALE[3], 0.0)
    return math.hypot(dx, dy)


def choca(p):
    return al_pale(p) < R_PALE


def experto(p, lado=+1):
    """El demostrador (un operario con el mando, por teleoperación): rodea el palé por arriba (lado = +1) o por abajo
    (−1). Va a la esquina de antes del palé, pasa a la altura Y_PASO hasta la de después y de ahí va a la meta, con
    pasos de 5 cm."""
    if p[0] < X_ANTES:
        objetivo = np.array([X_ANTES, lado * Y_PASO])
    elif p[0] < X_DESPUES:
        objetivo = np.array([X_DESPUES, lado * Y_PASO])
    else:
        objetivo = META
    return PASO * unitario(objetivo - p)


def rodar(politica, p0, n_pasos=250, empuje=None, empuje_en=22):
    """La política en lazo cerrado desde p0. 'empuje' (m, en y) es un golpe que desplaza el robot en el paso
    'empuje_en' (a los 2,2 s, cuando ya sube hacia la esquina: alguien lo aparta, una rueda resbala). Devuelve la
    trayectoria y cómo acaba: 'llega' (a menos de 10 cm de la meta), 'choca', 'se sale' o 'no llega'."""
    p = np.array(p0, float)
    traza = [p.copy()]
    for k in range(n_pasos):
        p = p + politica(p)
        if empuje is not None and k == empuje_en:
            p = p + np.array([0.0, empuje])
        traza.append(p.copy())
        if choca(p):
            return np.array(traza), 'choca'
        if np.linalg.norm(p - META) < 0.1:
            return np.array(traza), 'llega'
        if np.abs(p - [2.0, 0.0]).max() > 3.5:
            return np.array(traza), 'se sale'
    return np.array(traza), 'no llega'


SIGMA_DEMO = 0.004                       # el ruido del mando, por componente (m por paso)


def demostraciones(n=40, lado=+1, semilla=40, sigma=SIGMA_DEMO):
    """n demostraciones por teleoperación desde salidas repartidas cerca de (0; 0): pares (posición, acción)."""
    rng = np.random.default_rng(semilla)
    X, Y, trazas = [], [], []
    for _ in range(n):
        p = np.array([rng.normal(0, 0.05), rng.normal(0, 0.10)])
        traza = [p.copy()]
        for _ in range(250):
            a = experto(p, lado) + rng.normal(0, sigma, 2)
            X.append(p.copy()); Y.append(a.copy())
            p = p + a
            traza.append(p.copy())
            if np.linalg.norm(p - META) < 0.1:
                break
        trazas.append(np.array(traza))
    return np.array(X), np.array(Y), trazas


def ajustar_red(X, Y, semilla=40):
    """La clonación de comportamiento: una red de 64 × 64 (la de S17, más grande) de la posición a la acción."""
    from sklearn.neural_network import MLPRegressor
    return MLPRegressor(hidden_layer_sizes=(64, 64), activation='tanh', max_iter=1500, random_state=semilla,
                        tol=1e-9).fit(X, Y / PASO)


def como_politica(red):
    return lambda p: PASO * red.predict(np.asarray(p, float).reshape(1, -1))[0]


EMPUJONES = [0.0, -0.2, -0.4, -0.6, -0.8]     # hacia abajo, en m, en el paso 22


def dagger(X, Y, politica, empujones=EMPUJONES, lado=+1):
    """Una ronda de DAgger (Ross, Gordon y Bagnell, 2011): la copia conduce, con los empujones; el demostrador dice
    qué habría hecho en cada sitio por el que pasó; se añaden esos pares y se reentrena."""
    nX, nY = [], []
    for e in empujones:
        traza, _ = rodar(politica, np.array([0.0, 0.0]), empuje=e)
        for q in traza:
            nX.append(q); nY.append(experto(q, lado))
    X, Y = np.vstack([X, nX]), np.vstack([Y, nY])
    return X, Y, como_politica(ajustar_red(X, Y))


def tasa(politica, n=11, rango=0.15):
    """n salidas desde (0; y0), con y0 de −rango a +rango: cuántas llegan, chocan, se salen o no llegan."""
    res = {}
    for y0 in np.linspace(-rango, rango, n):
        _, fin = rodar(politica, np.array([0.0, y0]))
        res[fin] = res.get(fin, 0) + 1
    return res


def angulo(a):
    return math.degrees(math.atan2(a[1], a[0]))


# La difusión en miniatura: una red aprende a quitar el ruido que se añadió a una acción (50 niveles), sabiendo dónde
# está el robot. Para actuar, se parte de ruido y se le quita 50 veces. Solo scikit-learn, con semilla.
T_DIF = 50
BETA = np.linspace(1e-4, 0.2, T_DIF)
ALFA_D = 1 - BETA
ABAR = np.cumprod(ALFA_D)


def nivel(t):
    """El nivel de ruido t, escrito como 8 números que la red entiende (senos y cosenos)."""
    k = np.arange(4)
    e = (np.asarray(t)[:, None] / T_DIF) * np.exp(-k * np.log(1000) / 4)[None]
    return np.c_[np.sin(e), np.cos(e)]


def entrenar_difusion(X, Y, repeticiones=10, capas=(64, 64), epocas=60, semilla=0):
    """Cada demostración (posición, acción en módulo 1), con 'repeticiones' niveles de ruido al azar: la red aprende
    a predecir el ruido a partir de la acción ruidosa, la posición y el nivel."""
    from sklearn.neural_network import MLPRegressor
    rng = np.random.default_rng(semilla)
    A = Y / np.linalg.norm(Y, axis=1, keepdims=True)
    idx = np.repeat(np.arange(len(X)), repeticiones)
    t = rng.integers(0, T_DIF, len(idx))
    ruido = rng.standard_normal((len(idx), 2))
    xt = np.sqrt(ABAR[t])[:, None] * A[idx] + np.sqrt(1 - ABAR[t])[:, None] * ruido
    red = MLPRegressor(hidden_layer_sizes=capas, activation='relu', learning_rate_init=2e-3, batch_size=256,
                       max_iter=epocas, random_state=semilla, tol=0, n_iter_no_change=1000)
    return red.fit(np.c_[xt, X[idx], nivel(t)], ruido)


def muestrear(red, p, n=200, semilla=1):
    """n acciones para la posición p: se parte de ruido y se le quita, nivel a nivel, el que la red predice."""
    r = np.random.default_rng(semilla)
    x = r.standard_normal((n, 2))
    C = np.repeat(np.atleast_2d(p), n, 0)
    for k in reversed(range(T_DIF)):
        e = red.predict(np.c_[x, C, nivel(np.full(n, k))])
        x = (x - BETA[k] / np.sqrt(1 - ABAR[k]) * e) / np.sqrt(ALFA_D[k])
        if k > 0:
            x += np.sqrt(BETA[k]) * r.standard_normal(x.shape)
    return x


def inversa_2r(p, codo, a1=0.35, a2=0.20):
    """La cinemática inversa del 2R del puesto (S10), en grados: codo = +1, codo abajo; −1, codo arriba."""
    x, y = p
    c2 = (x * x + y * y - a1 * a1 - a2 * a2) / (2 * a1 * a2)
    q2 = codo * math.acos(c2)
    q1 = math.atan2(y, x) - math.atan2(a2 * math.sin(q2), a1 + a2 * math.cos(q2))
    return math.degrees(q1), math.degrees(q2)


def directa_2r(q1d, q2d, a1=0.35, a2=0.20):
    """Dónde queda la pinza (m) con los ángulos q1 y q2 en grados (S9)."""
    q1, q2 = math.radians(q1d), math.radians(q2d)
    return np.array([a1 * math.cos(q1) + a2 * math.cos(q1 + q2), a1 * math.sin(q1) + a2 * math.sin(q1 + q2)])


def media_circular(grados):
    """La media de unos rumbos (S19): la de sus vectores unitarios, no la de los números."""
    g = np.radians(grados)
    return math.degrees(math.atan2(np.sin(g).sum(), np.cos(g).sum()))


def de(res):
    """Cómo acaban las salidas, en texto: «llega 11 de 11»."""
    return ', '.join(f'{k} {v} de {sum(res.values())}' for k, v in res.items())


def dibuja_pale(ax, titulo=''):
    x, y, a, b = PALE
    ax.add_patch(plt.Rectangle((x - a - R_PALE, y - b - R_PALE), 2 * (a + R_PALE), 2 * (b + R_PALE), color='#F5D6D1', zorder=0))
    ax.add_patch(plt.Rectangle((x - a, y - b), 2 * a, 2 * b, color='#4A4F63', zorder=1))
    ax.plot(*META, '*', color=IQS_VERDE, ms=14, zorder=5)
    ax.set_xlim(-0.4, 4.4); ax.set_ylim(-1.7, 1.7); ax.set_aspect('equal')
    ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)
    ax.set_title(titulo, fontsize=10)


X, Y, trazas = demostraciones()
t0 = time.perf_counter()
red = ajustar_red(X, Y)
copia = como_politica(red)
error = float(np.sqrt(((red.predict(X) - Y / PASO) ** 2).sum(axis=1).mean())) * PASO
print(f'40 demostraciones: {len(X)} pares; la más cercana pasa a {coma(min(al_pale(q) for t in trazas for q in t))} m del palé')
print(f'la copia: error {coma(100 * error)} cm por paso; el del mando, {coma(100 * math.sqrt(2) * SIGMA_DEMO)} cm '
      f'(la red, en {coma(time.perf_counter() - t0, 1)} s)')
print('11 salidas:', de(tasa(copia)))

fig, ax = plt.subplots(figsize=(7, 3.3))
dibuja_pale(ax, '40 demostraciones (gris) y la copia desde 11 salidas (azul)')
for t in trazas:
    ax.plot(t[:, 0], t[:, 1], color=GRIS, lw=0.6)
for y0 in np.linspace(-0.15, 0.15, 11):
    tr, _ = rodar(copia, [0.0, y0])
    ax.plot(tr[:, 0], tr[:, 1], color=IQS_AZUL, lw=1.2)
plt.show()

## 2. Taller: el empujón (diapositivas 26 y 28)

A los 2,2 s (paso 22), cuando el robot ya sube hacia la esquina, alguien lo aparta EMPUJON metros hacia abajo. El operario se recupera: sabe qué hacer en cualquier sitio. La copia solo sabe lo que vio, y el empujón la deja donde ningún operario estuvo: allí se equivoca, y cada error la aleja un poco más. Es la **deriva**. La última línea mide cuánto se aleja de los datos: la distancia de la copia al par de demostración más cercano.

**Taller.** Ejecuta con EMPUJON = −0,4. Si te sobra tiempo, prueba −0,2 y −0,8.

In [ ]:
EMPUJON = -0.4      # prueba -0.2 y -0.8

vecinos = NearestNeighbors(n_neighbors=1).fit(X)
tc, fc = rodar(copia, [0.0, 0.0], empuje=EMPUJON)
te, fe = rodar(lambda p: experto(p, +1), [0.0, 0.0], empuje=EMPUJON)
print(f'copia: {fc}; operario: {fe}')
print(f'se aleja de los datos hasta {coma(vecinos.kneighbors(tc)[0].max())} m')

fig, ax = plt.subplots(figsize=(7, 3.3))
dibuja_pale(ax, f'Un empujón de {coma(EMPUJON, 1)} m a los 2,2 s')
ax.plot(X[:, 0], X[:, 1], '.', color=GRIS, ms=1, alpha=0.3, label='los datos')
ax.plot(te[:, 0], te[:, 1], color=GRIS, lw=2, label=f'operario: {fe}')
ax.plot(tc[:, 0], tc[:, 1], color=ROJO if fc == 'choca' else IQS_AZUL, lw=2, label=f'copia: {fc}')
if fc == 'choca':
    ax.plot(*tc[-1], 'x', color=ROJO, ms=12, mew=3)
ax.legend(fontsize=8, loc='lower right'); plt.show()

## 3. En casa: DAgger (diapositiva 26)

DAgger (Ross, Gordon y Bagnell, 2011) no pide más demostraciones del operario por su camino: deja conducir a la copia, con los empujones, y le pregunta al operario qué habría hecho **en cada sitio por el que pasó la copia**. Se añaden esos pares y se reentrena. Dos rondas (unos segundos).

In [ ]:
Xd, Yd, pol = X, Y, copia
for ronda in (1, 2):
    Xd, Yd, pol = dagger(Xd, Yd, pol)
    fines = [rodar(pol, [0.0, 0.0], empuje=e)[1] for e in EMPUJONES]
    print(f'ronda {ronda}: {len(Xd)} pares; con los empujones de 0 a −0,8 m: {", ".join(fines)}; 11 salidas: {de(tasa(pol))}')
copia_dagger = pol

## 4. Taller: la media de dos soluciones buenas es una mala (diapositivas 27 y 28)

Ahora hay dos operarios: uno rodea por arriba y otro por abajo, 20 viajes cada uno. Las dos son buenas. En la salida, uno sale a +35,4° y el otro a −35,4°, y la red, que minimiza el error cuadrático, aprende algo parecido a su **media**: un rumbo que ningún operario dio, más corto y hacia el palé.

La misma cuenta aparece dos veces más en el curso:
- **el 2R del puesto (S10):** la pieza en (0,4; 0,2) m tiene dos soluciones, codo abajo y codo arriba; la media de sus ángulos deja la pinza lejos, con el brazo estirado (la singularidad de S11);
- **cinco rumbos en torno a 180°, como la nube de S19:** la media de los números de 179°, −179°, 178°, −177° y 180° da 36,2°; la media circular, la de los vectores, −179,8° (en S19, con la nube hacia el oeste, 92,8° frente a 178,6°).

**Taller.** Ejecuta y compara las cuatro líneas con la diapositiva.

In [ ]:
Xa, Ya, ta = demostraciones(20, +1, semilla=1)          # por arriba
Xb, Yb, tb = demostraciones(20, -1, semilla=2)          # por abajo
Xm, Ym = np.vstack([Xa, Xb]), np.vstack([Ya, Yb])
mezcla = como_politica(ajustar_red(Xm, Ym))
p0 = np.array([0.0, 0.0])
print(f'operario: +{coma(angulo(experto(p0, +1)), 1)}° o {coma(angulo(experto(p0, -1)), 1)}°')
print(f'la red de los dos modos: {coma(angulo(mezcla(p0)), 1)}°')

P = (0.4, 0.2)
abajo, arriba = inversa_2r(P, +1), inversa_2r(P, -1)
media = ((abajo[0] + arriba[0]) / 2, (abajo[1] + arriba[1]) / 2)
print(f'el 2R: la pinza a {coma(1000 * np.linalg.norm(directa_2r(*media) - P), 1)} mm')
rumbos = [179, -179, 178, -177, 180]
print(f'cinco rumbos en torno a 180°: {coma(np.mean(rumbos), 1)}° frente a {coma(media_circular(rumbos), 1)}°')

tm, fm = rodar(mezcla, p0)
print(f'(el paso de la red, {coma(100 * np.linalg.norm(mezcla(p0)))} cm frente a 5; el 2R: codo abajo ({coma(abajo[0], 1)}°; '
      f'{coma(abajo[1], 1)}°), codo arriba ({coma(arriba[0], 1)}°; {coma(arriba[1], 1)}°), media ({coma(media[0], 1)}°; '
      f'{coma(media[1], 1)}°); en el lazo, la red de los dos modos: {fm}, a {coma(min(al_pale(q) for q in tm))} m del palé)')

fig, ax = plt.subplots(figsize=(7, 3.3))
dibuja_pale(ax, 'Dos operarios, y la red que los promedia')
for t in ta:
    ax.plot(t[:, 0], t[:, 1], color=IQS_AZUL, lw=0.5, alpha=0.6)
for t in tb:
    ax.plot(t[:, 0], t[:, 1], color=AMBAR, lw=0.5, alpha=0.6)
for a, c in ((experto(p0, +1), IQS_AZUL), (experto(p0, -1), AMBAR), (mezcla(p0), ROJO)):
    ax.annotate('', xy=p0 + 12 * a, xytext=p0, arrowprops=dict(arrowstyle='->', color=c, lw=2.5))
ax.plot(tm[:, 0], tm[:, 1], '--', color=ROJO, lw=1.5, label='la red de los dos modos, en el lazo')
ax.legend(fontsize=8, loc='lower right'); plt.show()

Una advertencia honesta: en este pasillo, la red de los dos modos acaba eligiendo un lado y llega (la última línea entre paréntesis). El fallo está en el principio: un paso más corto y torcido hacia el palé, que no dio ningún operario. Con un palé más ancho, de 0,8 × 1,8 m, la misma receta (20 + 20 demostraciones, la misma red) choca 2 de las 11 veces y se sale otras 2, mientras el operario y la copia de un solo lado llegan 11 de 11.

## 5. En casa: la difusión en miniatura (diapositiva 27)

La política de difusión no predice la acción: aprende a **quitar ruido**. En el entrenamiento, a cada acción de las demostraciones se le suma ruido de 50 niveles, y una red (de scikit-learn, 64 × 64) aprende a adivinar ese ruido sabiendo dónde está el robot y cuánto ruido hay. Para actuar, se parte de ruido puro y se le quita, nivel a nivel. Cada vez sale una acción distinta: unas por arriba y otras por abajo, y casi ninguna hacia el palé.

Entrena en unos 10 s.

In [ ]:
t0 = time.perf_counter()
difusion = entrenar_difusion(Xm, Ym)
print(f'(entrena en {coma(time.perf_counter() - t0, 1)} s)')
muestras = muestrear(difusion, [0.0, 0.0], 200)
ang = np.degrees(np.arctan2(muestras[:, 1], muestras[:, 0]))
print(f'la difusión, en (0; 0): {pc(np.mean(ang > 15))} por arriba, {pc(np.mean(ang < -15))} por abajo, '
      f'{pc(np.mean(np.abs(ang) <= 15))} hacia el palé')

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.hist(ang, bins=36, range=(-90, 90), color=IQS_VERDE, edgecolor='white', label='200 acciones de la difusión')
for v, c, t in ((angulo(experto(p0, +1)), IQS_AZUL, 'por arriba'), (angulo(experto(p0, -1)), AMBAR, 'por abajo'),
                (angulo(mezcla(p0)), ROJO, 'la red de los dos modos')):
    ax.axvline(v, color=c, lw=2, ls='--', label=t)
ax.set_xlabel('rumbo de la acción en (0; 0) (°)'); ax.set_ylabel('acciones')
ax.xaxis.set_major_formatter(COMA); ax.legend(fontsize=7); plt.show()

## 6. En casa: una orden, un VLA en miniatura (diapositiva 29)

El otro remedio es decirle cuál de las dos: se añade a la entrada un número, la orden: +1 es «por arriba» y −1, «por abajo». Con la orden, la acción correcta vuelve a ser una sola, y la misma red, con los mismos datos, ya no promedia. Es lo que hace un modelo visión-lenguaje-acción, en miniatura: la orden, en lugar de un número, es una frase.

In [ ]:
Xc = np.vstack([np.c_[Xa, np.full(len(Xa), 1.0)], np.c_[Xb, np.full(len(Xb), -1.0)]])
red_orden = ajustar_red(Xc, Ym)


def con_orden(m):
    return lambda p: PASO * red_orden.predict(np.append(p, m).reshape(1, -1))[0]


for m, nombre in ((1.0, 'por arriba'), (-1.0, 'por abajo')):
    print(f'con la orden «{nombre}»: {de(tasa(con_orden(m)))}; sale a {coma(angulo(con_orden(m)(p0)), 1)}°')

### Ejercicio 1 (en casa): el empujón en otro momento

DAgger solo ha visto empujones en el paso 22. Empuja en el paso 18 y en el 26 (`empuje_en=`) con los cinco empujones, y compara la copia, la de DAgger y el operario. ¿Choca alguna? ¿Por qué el operario no siempre se salva?

In [ ]:
# Ejercicio 1
for k in (18, 26):
    for nombre, p in (('copia', copia), ('DAgger', copia_dagger), ('operario', lambda q: experto(q, +1))):
        pass   # [rodar(p, [0.0, 0.0], empuje=e, empuje_en=k)[1] for e in EMPUJONES]

### Ejercicio 2 (en casa): una orden que no se dio nunca

Con la red de la sección 6, da la orden 0, que no está en los datos (ni «por arriba» ni «por abajo»). ¿Qué hace? ¿A qué rumbo sale? ¿Qué te dice de lo que entiende la red de la orden?

In [ ]:
# Ejercicio 2
pass   # tasa(con_orden(0.0)) y angulo(con_orden(0.0)(p0))

### Ejercicio 3 (en casa): ¿arregla la media circular la de los dos modos?

La media circular de S19 arregla la de los cinco rumbos en torno a 180° de la sección 4. Calcula con `media_circular` la de los dos operarios del palé (+35,4° y −35,4°). ¿Arregla la media de dos modos? ¿Qué problema arregla y cuál no?

In [ ]:
# Ejercicio 3
pass   # media_circular([...]) para los cinco rumbos de la sección 4 y para los dos operarios

---

## Soluciones

**Ejercicio 1.** En el paso 18, la copia choca con 0,4 m o más; la de DAgger y el operario llegan con los cinco. En el paso 26 (el robot ya en (1,06; 0,78), casi en la esquina), la copia vuelve a chocar desde 0,4 m; la de DAgger llega con todos menos con 0,8 m, y ahí también choca el operario: este operario es una regla sencilla (ir a la esquina en línea recta) y desde tan abajo y tan cerca, esa línea corta el palé. DAgger copia al operario con sus fallos: no puede ser mejor que él. Y lo que DAgger aprendió con empujones en el paso 22 sirve en el 18 y en el 26, porque lo que importa son los sitios por los que pasó, no el momento.

**Ejercicio 2.** Con la orden 0, choca las 11 veces: sale a −9,7°, hacia el palé, casi como la red de los dos modos (−10,4°). La red no sabe qué «significa» la orden: solo ha visto +1 y −1, y entre los dos interpola, que es otra vez la media. Lo que no se vio no se sabe, también en la orden: por eso los VLA se entrenan con muchísimas frases y tareas distintas.

**Ejercicio 3.** La media circular de los cinco rumbos en torno a 180° es −179,8°, bien; la de +35,4° y −35,4° es 0°: recto hacia el palé. La media circular arregla la vuelta de los ángulos (que 179° y −179° están al lado), no la de dos soluciones distintas: la media de dos modos sigue siendo un rumbo que nadie dio. Para eso hace falta modelar la distribución (la difusión) o dar la orden; los trozos de acciones ayudan a no cambiar de modo a mitad de camino.

---

## Para llevarse de esta sesión

**Copiar es aprendizaje supervisado, y copia lo que vio.** Con 40 viajes, la copia llega desde las 11 salidas; con un empujón de 0,4 m, se va a donde nadie estuvo y choca. DAgger pregunta al operario por los sitios de la copia, y en dos rondas llega con los cinco empujones.

**La media de dos soluciones buenas es una mala.** El 2R de S10 con la pinza a 102,8 mm, cinco rumbos en torno a 180° y el palé (+35,4° y −35,4°: la media, 0°; la red, −10,4°). La difusión saca acciones de los dos lados; la orden dice cuál.

**Un VLA es esto con una frase por orden y muchos robots por datos,** y lo que no se vio, tampoco lo sabe.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*